This notebook creates all plots regarding results on datasets containing dose-response curves

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)
from scipy.optimize import curve_fit
from src.simulate_func import hill_func
from collections import Counter
from matplotlib.lines import Line2D

### Plot dose response curves per cell type using the fitted parameter set

In [2]:
# Load simulation data 
df_sim = pd.read_csv("results/simulations_IL2s_fit.csv")
df_sim = df_sim[df_sim.columns[1:]]

# Load in vitro data
df_invit = pd.read_csv("data/signaling/IL2_data.csv")

# Directory location where to store plots
directory = "figures/fit_param/"

# Colors for mutants:
color_sim = ["blue","darkgreen"]
color_invit = ["cornflowerblue","forestgreen"]
markers = ['o','^']
linestyles = ["dotted","dashdot"]

In [3]:
for plot_num in list(dict.fromkeys(df_sim["Plot"])):
    df_sim_plot = df_sim.loc[df_sim["Plot"]==plot_num]
    df_invit_plot = df_invit.loc[df_invit["Plot"]==plot_num]

    # First plot WT
    variant = list(dict.fromkeys(df_sim_plot["Variant"]))[0]
    fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
    max_WT = df_sim_plot.loc[df_sim_plot["Variant"]==variant]["Result"].max()
    ax.scatter(df_invit_plot.loc[df_invit_plot["Variant"]==variant]["IL"].values, df_invit_plot.loc[df_invit_plot["Variant"]==variant]["pSTAT"].values, s=200, linewidth=2, color="salmon",marker='*')
    ax.plot(df_sim_plot.loc[df_sim_plot["Variant"]==variant]["IL0"].values, np.multiply(df_sim_plot.loc[df_sim_plot["Variant"]==variant]["Result"].values,100/(max_WT)), linewidth=7.5, label=variant,color="darkred", linestyle="dashed")
    i=0
    # Then plot the other variants
    for variant in list(dict.fromkeys(df_sim_plot["Variant"]))[1:]:
        ax.scatter(df_invit_plot.loc[df_invit_plot["Variant"]==variant]["IL"].values, df_invit_plot.loc[df_invit_plot["Variant"]==variant]["pSTAT"].values, s=200, linewidth=2, color=color_invit[i],marker=markers[i])
        ax.plot(df_sim_plot.loc[df_sim_plot["Variant"]==variant]["IL0"].values, np.multiply(df_sim_plot.loc[df_sim_plot["Variant"]==variant]["Result"].values,100/(max_WT)), linewidth=7.5, label=variant,color=color_sim[i], linestyle=linestyles[i])
        i += 1
    plt.legend(loc="upper left", fontsize=20)
    plt.xscale('log')
    ax.set_xticks(10**np.arange(round(np.log10(df_sim_plot["IL0"].min()),0)+1,round(np.log10(df_sim_plot["IL0"].max()),0)+1,2))
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=25)
    ax.tick_params(axis='y', labelsize=25)
    cell = df_sim_plot["Cell_type"].values[0]
    if cell == "Treg":
        cell = "Regulatory T cells"
    elif cell == "T8":
        cell = "CD8+ T cells"
    elif cell == "T8.naive":
        cell = "Naïve CD8+ T cells"
    elif cell == "NK":
        cell = "NK cells"
    elif cell == "T4.M_IL2RAlow":
        cell = "Memory CD4+ T cells"
    elif cell == "T8.M":
        cell = "Memory CD8+ T cells"
    elif cell == "T8.M":
        cell = "Memory CD8+ T cells"
    elif cell == "nTregs":
        cell = "Naïve regulatory T cells"
    elif cell == "T4.naive_IL2RAlow":
        cell = "Naïve CD4+ T cells"
    elif cell == "T4_IL2RAlow":
        cell = "CD4+ T cells"
    elif cell == "mTregs":
        cell = "Memory regulatory T cells"
    
    plt.title(cell, fontsize=25)
    plt.ylabel(list(df_sim_plot["STAT_type"])[0] +' (%)', fontsize=25)
    plt.xlabel('IL-2 (M)', fontsize=25)
    plt.savefig(directory +'Plot_n'+str(df_sim_plot["Plot"].values[0])+'_cell_'+df_sim_plot["Cell_type"].values[0]+'.pdf', transparent=True, bbox_inches="tight")
    plt.close()

### Plot EC50 and amplitude per cell type/variant in a single plot for the training and test datasets

In [13]:
# Load simulation data 
df_sim = pd.read_csv("results/simulations_IL2s_fit.csv")
df_EC50_AMP = pd.read_csv('data/signaling/IL2_data_hill.csv')

In [14]:
# Normalize data to maximum pSTAT5 of WT dose-response curve
for plot_num in df_sim["Plot"].drop_duplicates():
    max_WT = df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]=="WT"),"Result"].max()
    df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]=="WT"),"Result"] = df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]=="WT"),"Result"]/max_WT*100
    for variant in df_sim.loc[(df_sim["Plot"]==plot_num),"Variant"].drop_duplicates()[1:]:
        df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]==variant),"Result"] = df_sim.loc[(df_sim["Plot"]==plot_num)&(df_sim["Variant"]==variant),"Result"]/max_WT*100

In [15]:
# Get absolute error from the training and test data
cell_list = ["T8","T8.naive","Tregs","NK","T4_IL2RAlow","ACH-000061"]
for cell in df_EC50_AMP["Cell_type"].drop_duplicates():
    for variant in df_sim.loc[df_sim["Cell_type"]==cell,"Variant"].drop_duplicates():
        for plot_num in df_sim.loc[(df_sim["Cell_type"]==cell)&(df_sim["Variant"]==variant),"Plot"].drop_duplicates():
            df_var = df_sim.loc[(df_sim["Cell_type"]==cell)&(df_sim["Variant"]==variant)&(df_sim["Plot"]==plot_num)]
            fit, cov = curve_fit(hill_func, df_var["IL0"], df_var["Result"].values, bounds = ([0,df_var["IL0"].min()/10], [110, df_var["IL0"].max()*10]))
            df_EC50_AMP.loc[(df_EC50_AMP["Cell_type"]==cell)&(df_EC50_AMP["Variant"]==variant)&(df_EC50_AMP["Plot"]==plot_num),"Amp_m"] = fit[0]
            df_EC50_AMP.loc[(df_EC50_AMP["Cell_type"]==cell)&(df_EC50_AMP["Variant"]==variant)&(df_EC50_AMP["Plot"]==plot_num),"EC50_m"] = np.log10(fit[1])

In [16]:
# Plot training data EC50
cell_list = ["T8","T8.naive","Tregs","NK","T4_IL2RAlow","ACH-000061"]
cell_list_print = ["CD8+ T cells", "Naïve CD8+ T cells", "Tregs", "NK cells", "CD4+ T cells","HH T cells"]
colors_var = {'WT':"darkred",'NKTR-358':"darkblue","MDNA11":"forestgreen","ALKS4230":"darkorange","THOR-707":"purple"}
fig,ax=plt.subplots(1,1,figsize=(9, 8), dpi=400)
for variant in df_sim["Variant"].drop_duplicates():
    EC50_data = []
    EC50_model = []
    for cell in cell_list:
        if cell in df_EC50_AMP.loc[(df_EC50_AMP["Variant"]==variant),"Cell_type"].drop_duplicates().values:
            if cell == "T8.naive" and variant == "WT":
                EC50_data.append(np.nan)
                EC50_model.append(np.nan)
            elif cell == "ACH-000061" and variant == "WT":
                EC50_data.append(np.nan)
                EC50_model.append(np.nan)
            else:
                EC50_data.append(df_EC50_AMP.loc[(df_EC50_AMP["Cell_type"]==cell)&(df_EC50_AMP["Variant"]==variant),"EC50_t"].values[0])
                EC50_model.append(df_EC50_AMP.loc[(df_EC50_AMP["Cell_type"]==cell)&(df_EC50_AMP["Variant"]==variant),"EC50_m"].values[0])
        else:
            EC50_data.append(np.nan)
            EC50_model.append(np.nan)
    ax.scatter(cell_list_print, EC50_model, s=200, linewidth=3, color=colors_var[variant], marker='o')
    ax.scatter(cell_list_print, EC50_data, s=200, linewidth=3, color=colors_var[variant], marker='x')
# Legend 1: data type markers
marker_handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='black',
        linestyle='None',
        markersize=14,
        label='Model simulation'
    ),
    Line2D(
        [0], [0],
        marker='x',
        color='black',
        linestyle='None',
        markeredgewidth=3,
        markersize=14,
        label='Experimental data'
    )
]
legend_datatype = ax.legend(
    handles=marker_handles,
    title="Data type",
    loc="center",
    bbox_to_anchor=(1.22, 0.62),
    fontsize=16,
    title_fontsize=18,
)
ax.add_artist(legend_datatype)
# Legend 2: IL-10 variant colors
variant_handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='none',
        markerfacecolor=colors_var[variant],
        markeredgecolor=colors_var[variant],
        markersize=14,
        label=variant
    )
    for variant in df_sim["Variant"].drop_duplicates()
]
legend_variants = ax.legend(
    handles=variant_handles,
    title="IL-10 variant",
    loc="center",
    bbox_to_anchor=(1.22, 0.35),
    fontsize=16,
    title_fontsize=18,
)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.xticks(rotation=90, ha="right")
plt.ylabel('EC50 (M)', fontsize=25)
plt.ylim(-12.2, -5.8)
plt.savefig(directory+'EC50_train.pdf', transparent=True, bbox_inches="tight", bbox_extra_artists=(legend_datatype, legend_variants), pad_inches=0.25)
plt.close()

In [18]:
# Plot training data amplitude
fig,ax=plt.subplots(1,1,figsize=(10, 8), dpi=400)
for variant in df_sim["Variant"].drop_duplicates():
    Amp_data = []
    Amp_model = []
    for cell in cell_list:
        if cell in df_EC50_AMP.loc[(df_EC50_AMP["Variant"]==variant),"Cell_type"].drop_duplicates().values:
            if cell == "T8.naive" and variant == "WT":
                Amp_data.append(np.nan)
                Amp_model.append(np.nan)
            elif cell == "ACH-000061" and variant == "WT":
                Amp_data.append(np.nan)
                Amp_model.append(np.nan)
            else:
                Amp_data.append(df_EC50_AMP.loc[(df_EC50_AMP["Cell_type"]==cell)&(df_EC50_AMP["Variant"]==variant),"Amp_t"].values[0])
                Amp_model.append(df_EC50_AMP.loc[(df_EC50_AMP["Cell_type"]==cell)&(df_EC50_AMP["Variant"]==variant),"Amp_m"].values[0])
        else:
            Amp_data.append(np.nan)
            Amp_model.append(np.nan)
    ax.scatter(cell_list_print, Amp_model, s=200, linewidth=3, color=colors_var[variant], marker='o')
    ax.scatter(cell_list_print, Amp_data, s=200, linewidth=3, color=colors_var[variant], marker='x')
# Legend 1: data type markers
marker_handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='black',
        linestyle='None',
        markersize=14,
        label='Model simulation'
    ),
    Line2D(
        [0], [0],
        marker='x',
        color='black',
        linestyle='None',
        markeredgewidth=3,
        markersize=14,
        label='Experimental data'
    )
]
legend_datatype = ax.legend(
    handles=marker_handles,
    title="Data type",
    loc="center",
    bbox_to_anchor=(1.22, 0.62),
    fontsize=16,
    title_fontsize=18,
)
ax.add_artist(legend_datatype)
# Legend 2: IL-10 variant colors
variant_handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='none',
        markerfacecolor=colors_var[variant],
        markeredgecolor=colors_var[variant],
        markersize=14,
        label=variant
    )
    for variant in df_sim["Variant"].drop_duplicates()
]
legend_variants = ax.legend(
    handles=variant_handles,
    title="IL-10 variant",
    loc="center",
    bbox_to_anchor=(1.22, 0.35),
    fontsize=16,
    title_fontsize=18,
)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.xticks(rotation=90, ha="right")
plt.ylabel('Amplitude (pSTAT5 %)', fontsize=25)
plt.ylim(81, 116)
plt.savefig(directory+'Amplitude_train.pdf', transparent=True, bbox_inches="tight", bbox_extra_artists=(legend_datatype, legend_variants), pad_inches=0.25)
plt.close()

In [19]:
# Plot test data EC50
cell_list = ["T8.M", "T8.naive", "T4.M_IL2RAlow", "T4.naive_IL2RAlow", "mTregs", "nTregs","ACH-000061"]
cell_list_print = ["Mem. CD8+ T cells", "Naïve CD8+ T cells", "Mem. CD4+ T cells", "Naïve CD4+ T cells","Mem. Tregs", "Naïve Tregs","HH T cells"]
colors_var = {'WT':"darkred",'NKTR-358':"darkblue","MDNA11":"forestgreen","ALKS4230":"darkorange","THOR-707":"purple"}
fig,ax=plt.subplots(1,1,figsize=(10, 8), dpi=400)
variant = "WT"
EC50_data = []
EC50_model = []
for cell in cell_list:
    if cell in df_EC50_AMP.loc[(df_EC50_AMP["Variant"]==variant),"Cell_type"].drop_duplicates().values:
        EC50_data.append(df_EC50_AMP.loc[(df_EC50_AMP["Cell_type"]==cell)&(df_EC50_AMP["Variant"]==variant),"EC50_t"].values[0])
        EC50_model.append(df_EC50_AMP.loc[(df_EC50_AMP["Cell_type"]==cell)&(df_EC50_AMP["Variant"]==variant),"EC50_m"].values[0])
    else:
        EC50_data.append(np.nan)
        EC50_model.append(np.nan)
ax.scatter(cell_list_print, EC50_model, s=200, linewidth=3, color=colors_var[variant], marker='o')
ax.scatter(cell_list_print, EC50_data, s=200, linewidth=3, color=colors_var[variant], marker='x')
# Legend 1: data type markers
marker_handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='black',
        linestyle='None',
        markersize=14,
        label='Model simulation'
    ),
    Line2D(
        [0], [0],
        marker='x',
        color='black',
        linestyle='None',
        markeredgewidth=3,
        markersize=14,
        label='Experimental data'
    )
]

legend_datatype = ax.legend(
    handles=marker_handles,
    title="Data type",
    loc="center",
    bbox_to_anchor=(1.22, 0.5),
    fontsize=16,
    title_fontsize=18,
)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.xticks(rotation=90, ha="right")
plt.ylabel('EC50 (M)', fontsize=25)
plt.ylim(-12.2, -5.8)
plt.savefig(directory+'EC50_test.pdf', transparent=True, bbox_inches="tight", bbox_extra_artists=(legend_datatype, legend_variants), pad_inches=0.25)
plt.close()

In [20]:
# Plot test data amplitude
fig,ax=plt.subplots(1,1,figsize=(10, 8), dpi=400)
variant = "WT"
Amp_data = []
Amp_model = []
for cell in cell_list:
    if cell in df_EC50_AMP.loc[(df_EC50_AMP["Variant"]==variant),"Cell_type"].drop_duplicates().values:
        Amp_data.append(df_EC50_AMP.loc[(df_EC50_AMP["Cell_type"]==cell)&(df_EC50_AMP["Variant"]==variant),"Amp_t"].values[0])
        Amp_model.append(df_EC50_AMP.loc[(df_EC50_AMP["Cell_type"]==cell)&(df_EC50_AMP["Variant"]==variant),"Amp_m"].values[0])
    else:
        Amp_data.append(np.nan)
        Amp_model.append(np.nan)
ax.scatter(cell_list_print, Amp_model, s=200, linewidth=3, color=colors_var[variant], marker='o')
ax.scatter(cell_list_print, Amp_data, s=200, linewidth=3, color=colors_var[variant], marker='x')
# Legend 1: data type markers
marker_handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='black',
        linestyle='None',
        markersize=14,
        label='Model simulation'
    ),
    Line2D(
        [0], [0],
        marker='x',
        color='black',
        linestyle='None',
        markeredgewidth=3,
        markersize=14,
        label='Experimental data'
    )
]

legend_datatype = ax.legend(
    handles=marker_handles,
    title="Data type",
    loc="center",
    bbox_to_anchor=(1.22, 0.5),
    fontsize=16,
    title_fontsize=18,
)
ax.spines["bottom"].set_linewidth(4)
ax.spines["left"].set_linewidth(4)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_tick_params(width=5, length=10)
ax.yaxis.set_tick_params(width=5, length=10)
ax.tick_params(axis='x', labelsize=25)
ax.tick_params(axis='y', labelsize=25)
plt.xticks(rotation=90, ha="right")
plt.ylabel('Amplitude (pSTAT5 %)', fontsize=25)
plt.ylim(81, 116)
plt.savefig(directory+'Amplitude_test.pdf', transparent=True, bbox_inches="tight", bbox_extra_artists=(legend_datatype, legend_variants), pad_inches=0.25)
plt.close()